<a href="https://colab.research.google.com/github/PradipNichite/Youtube-Tutorials/blob/main/Spacy_Custom_NER_Youtube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U spacy-transformers 'spacy_alignments'
!python -m spacy download en_core_web_trf
!pip install transformers>=4.36.0 huggingface_hub>=0.19.0 accelerate>=0.25.0

import spacy
import torch
import cupy
import sys
import numpy as np
import scipy
import time
import scipy.special
from spacy.training import Example
from spacy.util import minibatch, compounding, filter_spans
from spacy.tokens import DocBin
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm
import random
import json
import warnings
warnings.filterwarnings('ignore')

print(f"spaCy version: {spacy.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CuPy version: {cupy.__version__}")

nlp = spacy.load("en_core_web_trf")
doc = nlp("Hello world")
print("✅ Tutto funzionante!")

In [ ]:
seed = int(time.time())
random.seed(seed)
print(f"Seed usato per questa run: {seed}")

In [ ]:
print("CuPy version:", cupy.__version__)
print("CUDA available:", cupy.cuda.is_available())
print("Torch GPU disponibile?", torch.cuda.is_available())
print("Torch versione:", torch.__version__)
print("CUDA supportata da torch:", torch.version.cuda)

In [ ]:
# ------------------------- DATASET 1 -----------------------
with open('/kaggle/input/datasets/.../dataset-review/medical_dataset_NER_clear.json', 'r') as file:
    dataset1 = json.load(file)

medical_dataset_1 = dataset1['DatasetDict']['medical_consultations']['Dataset']['data']


# ------------------------- DATASET 2 -----------------------
with open('/kaggle/input/datasets/.../dataset-review/zenodo_dataset.json', 'r') as file:
    dataset2 = json.load(file)

medical_dataset_2 = dataset2['DatasetDict']['medical_consultations']['Dataset']['data']

def split_train_test(data, train_size=750, test_size=250):
    data_shuffled = data.copy()
    random.shuffle(data_shuffled)
    train_data = data_shuffled[:train_size]
    remaining = data_shuffled[train_size:]
    test_data = random.sample(remaining, test_size)
    return train_data, test_data

print(f"Dataset 1: {len(medical_dataset_1)} esempi totali disponibili")
print(f"Dataset 2: {len(medical_dataset_2)} esempi totali disponibili")


In [5]:
def convert_to_spacy_format(dataset_split):
    spacy_data = []

    for example in dataset_split:
        temp_dict = {}
        tokens = example['tokens']
        ent_tags = example['ent_tags']

        text = " ".join(tokens)

        temp_dict['text'] = text
        temp_dict['entities'] = []

        current_pos = 0

        for i in range(len(tokens)):
            token = tokens[i]

            start_idx = current_pos
            end_idx = start_idx + len(token)

            if ent_tags[i] != 'O':
                extracted_token = text[start_idx:end_idx]
                if extracted_token != token:
                    print("sbagliato!")

                temp_dict['entities'].append((start_idx, end_idx, ent_tags[i]))

            current_pos = end_idx + 1

        spacy_data.append(temp_dict)

    return spacy_data

In [6]:
def build_doc_bin(dataset, name):
    """Converte un dataset in formato spaCy e lo salva su disco come file .spacy separato"""
    data = convert_to_spacy_format(dataset)
    nlp_test = spacy.blank("en")
    doc_bin = DocBin()
    for example in tqdm(data, desc=f"Building {name}"):
        text = example['text']
        labels = example['entities']
        doc = nlp_test.make_doc(text)
        ents = []
        for start, end, label in labels:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                print(f"[{name}] Skipping entity")
            else:
                ents.append(span)
        filtered_ents = filter_spans(ents)
        doc.ents = filtered_ents
        doc_bin.add(doc)
    doc_bin.to_disk(f"{name}.spacy")
    print(f"{name}: {len(data)} esempi salvati in {name}.spacy")


In [7]:
from spacy.scorer import Scorer
import os

results_file_1 = "/kaggle/working/results_dataset_1.txt"
results_file_2 = "/kaggle/working/results_dataset_2.txt"

# Puliamo eventuali file di risultati di run precedenti, cosi da ripartire puliti
for f in [results_file_1, results_file_2]:
    if os.path.exists(f):
        os.remove(f)

def evaluate_test_set(path, name, nlp_model, output_file=None):
    test_doc_bin = DocBin().from_disk(path)
    reference_docs = list(test_doc_bin.get_docs(nlp_model.vocab))

    examples = []
    for ref_doc in reference_docs:
        pred_doc = nlp_model(ref_doc.text)
        examples.append(Example(pred_doc, ref_doc))

    scorer = Scorer()
    scores = scorer.score(examples)

    # Costruisce l'output testuale
    lines = []
    lines.append(f"===== Risultati su {name} ({len(reference_docs)} esempi) =====")
    lines.append(f"Precision: {scores.get('ents_p')}")
    lines.append(f"Recall: {scores.get('ents_r')}")
    lines.append(f"F1: {scores.get('ents_f')}")
    lines.append("Per-type:")
    per_type = scores.get("ents_per_type") or {}
    for label, metrics in per_type.items():
        lines.append(f"  {label}: {metrics}")

    output_text = "\n".join(lines)

    # Stampa a schermo
    print("\n" + output_text)

    # Salva su file in append, cosi da accumulare i risultati di tutti i training
    if output_file is None:
        output_file = f"{name.replace(' ', '_')}_results.txt"
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(output_text + "\n\n")
    print(f"\n[Salvato in {output_file}]")

    return scores


In [ ]:
N_RUNS = 30

all_scores_1 = []
all_scores_2 = []

for run in range(1, N_RUNS + 1):
    print(f"\n{'='*25} TRAINING RUN {run}/{N_RUNS} {'='*25}\n")

    # ---- Split: 750 per dataset per train+val, 250 per dataset per test (sempre casuale) ----
    train_1, test_1 = split_train_test(medical_dataset_1)
    train_2, test_2 = split_train_test(medical_dataset_2)

    medical_dataset = train_1 + train_2

    print(f"Dataset 1: {len(train_1)} train, {len(test_1)} test")
    print(f"Dataset 2: {len(train_2)} train, {len(test_2)} test")
    print(f"Totale training (train+val): {len(medical_dataset)}")

    # ---- Conversione in formato spaCy ----
    training_data = convert_to_spacy_format(medical_dataset)

    nlp = spacy.blank("en")
    doc_bin = DocBin()

    for training_example in tqdm(training_data):
        text = training_example['text']
        labels = training_example['entities']
        doc = nlp.make_doc(text)
        ents = []
        for start, end, label in labels:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                print("Skipping entity")
            else:
                ents.append(span)
        filtered_ents = filter_spans(ents)
        doc.ents = filtered_ents
        doc_bin.add(doc)

    doc_bin.to_disk("train.spacy")

    build_doc_bin(test_1, "test_1")
    build_doc_bin(test_2, "test_2")

    # ---- Split train/dev (80/20) ----
    nlp = spacy.blank("en")

    input_path = Path("./train.spacy")
    train_path = Path("./train_split.spacy")
    dev_path = Path("./dev_split.spacy")

    doc_bin = DocBin().from_disk(input_path)
    docs = list(doc_bin.get_docs(nlp.vocab))
    random.shuffle(docs)

    split_idx = int(len(docs) * 0.8)
    train_docs = docs[:split_idx]
    dev_docs = docs[split_idx:]

    train_bin = DocBin()
    for doc in train_docs:
        train_bin.add(doc)
    train_bin.to_disk(train_path)

    dev_bin = DocBin()
    for doc in dev_docs:
        dev_bin.add(doc)
    dev_bin.to_disk(dev_path)

    print(f"Total docs: {len(docs)}")
    print(f"Train: {len(train_docs)} \u2192 {train_path}")
    print(f"Dev: {len(dev_docs)} \u2192 {dev_path}")

    # ---- Training ----
    !python -m spacy init fill-config /kaggle/input/datasets/.../config-ner/base_config.cfg config.cfg
    !python -m spacy train /kaggle/working/config.cfg --output ./ --paths.train ./train_split.spacy --paths.dev ./dev_split.spacy --gpu-id 0

    # ---- Valutazione ----
    nlp_trained = spacy.load("/kaggle/working/model-best")

    scores_1 = evaluate_test_set("test_1.spacy", f"Dataset 1 - Run {run}", nlp_trained, output_file=results_file_1)
    scores_2 = evaluate_test_set("test_2.spacy", f"Dataset 2 - Run {run}", nlp_trained, output_file=results_file_2)

    all_scores_1.append(scores_1)
    all_scores_2.append(scores_2)

print("\n\u2705 Completati tutti i 5 training. Risultati salvati in:")
print(f" - {results_file_1}")
print(f" - {results_file_2}")
